In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load the sales dataset
sales = pd.read_csv("../data/raw/sales_train_validation.csv")

# Select the same SKU and store
target = sales[
    (sales["item_id"] == "FOODS_3_090") &
    (sales["store_id"] == "CA_3")
].iloc[0]

# Extract daily sales
day_columns = [
    col for col in sales.columns
    if col.startswith("d_")
]

series = target[day_columns].astype(float).to_numpy()

print("Total observations:", len(series))
print("First 5 observations:", series[:5])
print("Last 5 observations:", series[-5:])

Total observations: 1913
First 5 observations: [108. 132. 102. 120. 106.]
Last 5 observations: [ 88.  77. 141. 139. 130.]


In [3]:

# Forecasting configuration
forecast_horizon = 28  # Predict the next 28 days
n_splits = 5            # Evaluate across 5 test periods

# Generate rolling validation splits
splits = []

for i in range(n_splits, 0, -1):
    test_end = len(series) - (i - 1) * forecast_horizon
    test_start = test_end - forecast_horizon
    train_end = test_start

    splits.append((train_end, test_start, test_end))

# Display the split details
for fold, (train_end, test_start, test_end) in enumerate(splits, start=1):
    print(f"Fold {fold}")
    print(f"  Training observations: {train_end}")
    print(f"  Test observations: {test_end - test_start}")
    print(f"  Training ends at index: {train_end - 1}")
    print(f"  Test indices: {test_start} to {test_end - 1}")
    print()


Fold 1
  Training observations: 1773
  Test observations: 28
  Training ends at index: 1772
  Test indices: 1773 to 1800

Fold 2
  Training observations: 1801
  Test observations: 28
  Training ends at index: 1800
  Test indices: 1801 to 1828

Fold 3
  Training observations: 1829
  Test observations: 28
  Training ends at index: 1828
  Test indices: 1829 to 1856

Fold 4
  Training observations: 1857
  Test observations: 28
  Training ends at index: 1856
  Test indices: 1857 to 1884

Fold 5
  Training observations: 1885
  Test observations: 28
  Training ends at index: 1884
  Test indices: 1885 to 1912



In [16]:

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Store results for every model and fold
validation_results = []


def calculate_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    # Weighted Absolute Percentage Error
    denominator = np.abs(actual).sum()

    if denominator > 0:
        wape = (
            np.abs(actual - predicted).sum()
            / denominator
        ) * 100
    else:
        wape = np.nan

    return mae, rmse, wape



# Evaluate all five validation folds
for fold, (train_end, test_start, test_end) in enumerate(splits, start=1):

    train = np.asarray(series[:train_end], dtype=float)
    actual = np.asarray(series[test_start:test_end], dtype=float)

    # 1. Naive: predict using the latest training value
    naive_forecast = np.repeat(train[-1], len(actual))

    # 2. Seasonal Naive: repeat the previous 7-day pattern
    seasonal_forecast = np.resize(train[-7:], len(actual))

    # 3. Holt-Winters with additive trend and seasonality
    hw_forecast = ExponentialSmoothing(
        train,
        trend="add",
        seasonal="add",
        seasonal_periods=7,
        initialization_method="estimated"
    ).fit(optimized=True).forecast(len(actual))

    # 4. Damped Holt-Winters
    damped_forecast = ExponentialSmoothing(
        train,
        trend="add",
        damped_trend=True,
        seasonal="add",
        seasonal_periods=7,
        initialization_method="estimated"
    ).fit(optimized=True).forecast(len(actual))

    forecasts = {
        "Naive": naive_forecast,
        "Seasonal Naive": seasonal_forecast,
        "Holt-Winters": hw_forecast,
        "Damped Holt-Winters": damped_forecast
    }

    for model_name, predicted in forecasts.items():
        validation_results.append({
        "Fold": fold,
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE (%)": wape,
        "Actual Sum": np.abs(actual).sum(),
        "Absolute Error Sum": np.abs(actual - predicted).sum()
})

# Show results for each fold
fold_results = pd.DataFrame(validation_results)

display(fold_results.round(2))


,Fold,Model,MAE,RMSE,WAPE (%),Actual Sum,Absolute Error Sum
0,1,Naive,60.79,63.32,55.09,47.0,47.00
1,1,Seasonal Naive,60.79,63.32,55.09,47.0,865.00
2,1,Holt-Winters,60.79,63.32,55.09,47.0,481.78
3,1,Damped Holt-Winters,60.79,63.32,55.09,47.0,479.46
4,2,Naive,60.79,63.32,55.09,2379.0,2379.00
5,2,Seasonal Naive,60.79,63.32,55.09,2379.0,2367.00
6,2,Holt-Winters,60.79,63.32,55.09,2379.0,2754.77
7,2,Damped Holt-Winters,60.79,63.32,55.09,2379.0,2711.71
8,3,Naive,60.79,63.32,55.09,3042.0,860.00
9,3,Seasonal Naive,60.79,63.32,55.09,3042.0,772.00


In [14]:

# Average performance across all five folds
summary = (
    fold_results
    .groupby("Model")[["MAE", "RMSE", "WAPE (%)"]]
    .mean()
    .sort_values("MAE")
)

print("Average performance across 5 validation folds:")
display(summary.round(2))


Average performance across 5 validation folds:


,MAE,RMSE,WAPE (%)
Model,,,
Seasonal Naive,42.06,52.51,405.20
Damped Holt-Winters,45.58,50.36,247.50
Holt-Winters,46.04,50.92,248.97
Naive,50.81,57.03,70.37


In [17]:
# Calculate pooled WAPE across all validation folds
# using the original fold-level actual and predicted values.

pooled_results = []

for model_name in fold_results["Model"].unique():
    model_folds = fold_results[
        fold_results["Model"] == model_name
    ]

    # Check whether actual and predicted totals are available
    if {"Actual Sum", "Absolute Error Sum"}.issubset(
        model_folds.columns
    ):
        actual_total = model_folds["Actual Sum"].sum()
        absolute_error_total = model_folds[
            "Absolute Error Sum"
        ].sum()

        pooled_wape = (
            absolute_error_total / abs(actual_total) * 100
            if actual_total != 0
            else float("nan")
        )

        pooled_results.append({
            "Model": model_name,
            "Pooled WAPE (%)": pooled_wape
        })

if pooled_results:
    pooled_summary = pd.DataFrame(pooled_results)
    display(pooled_summary.round(2))
else:
    print(
        "The fold_results dataframe does not contain "
        "'Actual Sum' and 'Absolute Error Sum'."
    )
    print("Next, inspect the cell that creates fold_results.")

,Model,Pooled WAPE (%)
0,Naive,60.93
1,Seasonal Naive,50.43
2,Holt-Winters,55.20
3,Damped Holt-Winters,54.65


In [11]:
print("Columns in fold_results:")
print(fold_results.columns.tolist())

print("\nFirst 10 rows:")
display(fold_results.head(10))

print("\nData types:")
print(fold_results.dtypes)

Columns in fold_results:
['Fold', 'Model', 'MAE', 'RMSE', 'WAPE (%)']

First 10 rows:


,Fold,Model,MAE,RMSE,WAPE (%)
0,1,Naive,1.678571,2.994042,100.000000
1,1,Seasonal Naive,30.892857,47.267098,1840.425532
2,1,Holt-Winters,17.206384,20.254356,1025.061162
3,1,Damped Holt-Winters,17.123722,19.689480,1020.136617
4,2,Naive,84.964286,95.090371,100.000000
5,2,Seasonal Naive,84.535714,94.647655,99.495586
6,2,Holt-Winters,98.384553,104.191562,115.795186
7,2,Damped Holt-Winters,96.846760,102.686591,113.985257
8,3,Naive,30.714286,37.196390,28.270874
9,3,Seasonal Naive,27.571429,33.157632,25.378041



Data types:
Fold          int64
Model        object
MAE         float64
RMSE        float64
WAPE (%)    float64
dtype: object


In [7]:

# Inspect actual demand in each validation fold
for fold, (train_end, test_start, test_end) in enumerate(splits, start=1):
    actual = np.asarray(series[test_start:test_end], dtype=float)

    print(
        f"Fold {fold}: "
        f"mean demand = {actual.mean():.2f}, "
        f"minimum = {actual.min():.2f}, "
        f"total demand = {actual.sum():.2f}"
    )


Fold 1: mean demand = 1.68, minimum = 0.00, total demand = 47.00
Fold 2: mean demand = 84.96, minimum = 0.00, total demand = 2379.00
Fold 3: mean demand = 108.64, minimum = 55.00, total demand = 3042.00
Fold 4: mean demand = 111.36, minimum = 62.00, total demand = 3118.00
Fold 5: mean demand = 110.36, minimum = 61.00, total demand = 3090.00


In [8]:

# Average performance across all validation folds
summary = (
    fold_results
    .groupby("Model")[["MAE", "RMSE", "WAPE (%)"]]
    .mean()
    .sort_values("MAE")
)

display(summary.round(2))


,MAE,RMSE,WAPE (%)
Model,,,
Seasonal Naive,42.06,52.51,405.20
Damped Holt-Winters,45.58,50.36,247.50
Holt-Winters,46.04,50.92,248.97
Naive,50.81,57.03,70.37


In [12]:
print("Number of validation folds:", len(splits))
print("Available models:", fold_results["Model"].unique())

display(
    fold_results.sort_values(["Model", "Fold"]).round(2)
)

Number of validation folds: 5
Available models: ['Naive' 'Seasonal Naive' 'Holt-Winters' 'Damped Holt-Winters']


,Fold,Model,MAE,RMSE,WAPE (%)
3,1,Damped Holt-Winters,17.12,19.69,1020.14
7,2,Damped Holt-Winters,96.85,102.69,113.99
11,3,Damped Holt-Winters,25.49,33.48,23.46
15,4,Damped Holt-Winters,27.65,32.65,24.83
19,5,Damped Holt-Winters,60.79,63.32,55.09
2,1,Holt-Winters,17.21,20.25,1025.06
6,2,Holt-Winters,98.38,104.19,115.80
10,3,Holt-Winters,25.92,33.99,23.86
14,4,Holt-Winters,27.58,32.56,24.77
18,5,Holt-Winters,61.10,63.62,55.36


In [10]:

# Combine average fold metrics with pooled WAPE
final_comparison = summary.join(
    pooled_summary.set_index("Model")
)

final_comparison = final_comparison.sort_values("MAE")

display(final_comparison.round(2))

# Save results for future analysis
final_comparison.to_csv(
    "../data/processed/validation_model_comparison.csv"
)

print("Model comparison saved successfully.")


,MAE,RMSE,WAPE (%),Pooled WAPE (%)
Model,,,,
Seasonal Naive,42.06,52.51,405.20,50.43
Damped Holt-Winters,45.58,50.36,247.50,54.65
Holt-Winters,46.04,50.92,248.97,55.20
Naive,50.81,57.03,70.37,60.93


Model comparison saved successfully.
